In [337]:
import numpy as np
from auxiliares import cargar_datos_csv

In [338]:
class layer():
    """Clase capa:
    
    atributos de clase:
        n_neuronas, n_inputs, weights, outputs, activate, activate_derivative, deltas"""

    def __init__(self, n_neurons, n_inputs, activation_function):
        self.n_neurons = n_neurons
        self.n_inputs = n_inputs
        self.weights = np.random.uniform(-0.5, 0.5, (self.n_neurons, self.n_inputs+1))  

        match activation_function:
            case 'sigmoid':
                self.activate = self.sigmoid # 0,1
                self.activate_derivative = self.sigmoid_derivative
            case 'relu':
                self.activate = self.relu
                self.activate_derivative = self.relu_derivative
            case 'symmetry sigmoid':
                self.activate = self.symmetry_sigmoid # -1,1
                self.activate_derivative = self.symmetry_sigmoid_derivative
            case 'identity':
                self.activate = self.identity
                self.activate_derivative = self.identity_derivative
            case 'sign':
                self.activate = self.sign
                self.activate_derivative = self.sign_derivative
           
    def forward(self, inputs):
        """
        recibe un input con bias incluido como columna
        """
        self.inputs = inputs #ya viene con el bias incluido!?!??!?!?!?!?!?!? Si o No, diseño.
        self.outputs = self.activate(self.weights @ self.inputs)
        return self.outputs #retorna columna
    

    def calculate_deltas(self, error):
        self.deltas = error * self.activate_derivative(self.outputs)
        return self.deltas #retorna columna

    
    
    ##funciones auxiliares
    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x)) #0.1
    
    def sigmoid_derivative(self, x):
        return self.sigmoid(x) * (1 - self.sigmoid(x))
    
    def relu(self, x):
        return np.maximum(0, x)
    def relu_derivative(self, x):
        return np.where(x > 0, 1, 0)
    
    def symmetry_sigmoid(self, x):
        return 2 / (1 + np.exp(-x)) - 1
    def symmetry_sigmoid_derivative(self, x):
        return 0.5 * (1 + self.symmetry_sigmoid(x)) * (1 - self.symmetry_sigmoid(x))
    
    def identity(self, x):
        return x
    def identity_derivative(self, x):
        return np.ones_like(x)
    
    def sign(self, x):
        return np.where(x >= 0, 1, -1)
    def sign_derivative(self, x):
        return np.zeros_like(x)
    


In [339]:
class neural_network():
    def __init__(self, layers_config, size_input, max_epoch=1000, learning_rate =0.01, error_threshold=1e-6):
        """"Layers config es una lista de tuplas (n_neurons, activation_function)"""
        self.layers = []
        for i in range(len(layers_config)):
            neurons, activation = layers_config[i]
            self.layers.append(
                layer(
                    n_neurons = neurons,
                    n_inputs = size_input  if i==0 else layers_config[i-1][0], #entrada o cantidad de neuronas anterior
                    activation_function = activation
                )
            )
        self.max_epoch = max_epoch
        self.learning_rate = learning_rate
        self.error_threshold = error_threshold

    def fit(self, X, y):
        if y.ndim ==1:
            y = y.reshape(-1,1)
        
        #X = np.hstack((-1*np.ones((X.shape[0], 1)), X))
        for epoch in range(self.max_epoch):
            for i in range(X.shape[0]):
                x_current = np.array(X[i,:]).reshape(-1,1) #no le agregamos todavia el bias
                y_current = np.array(y[i,:]).reshape(-1,1) #pueden ser mas de una salida!!!!!!!
                self.forward_propagation(x_current) #lo pasamos como columna!!!!!!!!!!!!!!!
                self.backward_propagation(y_current)
                self._update_weights()

    
    def transform(self, X):
        y_pred = []
        for i in range(X.shape[0]):
            x_curr = np.array(X[i,:]).reshape(-1,1) #hacerlo columna
            y_pred.append(self.forward_propagation(x_curr))

        return np.array(y_pred).reshape(-1,1) 
         
    def forward_propagation(self, x):
        # recorrer capa por capa hasta llegar a la salida
        #siempre es vector
        for i, layer in enumerate(self.layers):
            if i == 0:
                x_bias = np.vstack((-1, x)) #agrego bias a la entrada
                self.layers[i].forward(x_bias) #lo guarda la capa el resultado
            else:
                x_bias = np.vstack((-1, self.layers[i-1].outputs)) #agrego bias al output de la capa anterior
                self.layers[i].forward(x_bias)

        return self.layers[-1].outputs
    
    def backward_propagation(self, y):
        #i = recorrer capa anterior, cantidad de entradas de actual
        #j recorrer capa actual
        # k recorrer todas la capa de salida

        for layer_index in range(len(self.layers)-1, -1,-1): #recorro todas las capas
            current_layer = self.layers[layer_index]
            if layer_index == len(self.layers)-1: #capa de salida
                error = current_layer.outputs - y #predicho - real
                
                current_layer.calculate_deltas(error) 
            else:
                next_layer = self.layers[layer_index + 1] #de aca saco el delta
                error_propagado = []

                for i in range(current_layer.n_neurons):  # i = neurona actual
                    suma = 0
                    # Sumar contribuciones de todas las neuronas de la SIGUIENTE capa
                    for j in range(next_layer.n_neurons):
                        # next_layer.weights[j, i+1] donde:
                        # j: neurona siguiente
                        # i+1: neurona actual (sesgo incluido)
                        suma += next_layer.deltas[j] * next_layer.weights[j, i+1]
                    error_propagado.append(suma)
                
                error_propagado = np.array(error_propagado).reshape(-1, 1)
                current_layer.calculate_deltas(error_propagado)

            
    def _update_weights(self):
        for layer_index in range(len(self.layers)):
            #recorro todas las capas
            current_layer = self.layers[layer_index]
            for j in range(current_layer.n_neurons):
                #recorro todas las neuronas de la capa actual
                for i in range(current_layer.n_inputs+1):
                    #producto punto
                    current_layer.weights[j,i] -= self.learning_rate * current_layer.deltas[j].item() * current_layer.inputs[i].item()


    def calculate_error(self, X, y):
        pass

    def score(self, X, y, threshold=0):

        #por ahora solo anda con la funcion signo de la sigmoide
        correct = 0
        total = len(X)

        for i in range(total):
            x = X[i].reshape(-1,1)   # columna
            output = self.forward_propagation(x) # salida sigmoide

            # decisión
            y_pred = 1 if output >= threshold else -1

            if y_pred == y[i]:
                correct += 1

        return correct / total


In [340]:
import os
import matplotlib.pyplot as plt

ruta_actual = os.getcwd()

#intentamos xor !!!!!!


X_train, y_train = cargar_datos_csv(os.path.join(ruta_actual, '../data/XOR_trn.csv'))
X_test, y_test = cargar_datos_csv(os.path.join(ruta_actual, '../data/XOR_tst.csv'))

n_netw= neural_network(
    layers_config = [(4, 'symmetry sigmoid'), (1, 'symmetry sigmoid')],
    size_input = X_train.shape[1],
    max_epoch=100,
    learning_rate=0.01,
    error_threshold=1e-6
)

n_netw.fit(X_train, y_train)

print(n_netw.score(X_test, y_test))

1.0
